In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# NB_Gold_Dim_Product
# Silver -> Gold
# ============================================================

SOURCE_TABLE = "silver_products"
TARGET_TABLE = "dim_product"

# ------------------------------------------------------------
# 1. Read Silver
# ------------------------------------------------------------

df = spark.table(SOURCE_TABLE)

# ------------------------------------------------------------
# 2. Select business attributes
# ------------------------------------------------------------

df = (
    df
    .select(
        "product_id",
        "product_name",
        "category",
        "price",
        "stock_quantity"
    )
)

# ------------------------------------------------------------
# 3. Generate surrogate key
# ------------------------------------------------------------

window = Window.orderBy("product_id")

df_gold = (
    df
    .withColumn(
        "product_key",
        F.row_number().over(window)
    )
    .select(
        "product_key",
        "product_id",
        "product_name",
        "category",
        "price",
        "stock_quantity"
    )
)

# ------------------------------------------------------------
# 4. Write Gold dimension
# ------------------------------------------------------------

df_gold.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(TARGET_TABLE)

# ------------------------------------------------------------
# 5. Validation
# ------------------------------------------------------------

print(f"Gold table created: {TARGET_TABLE}")
print(f"Records: {df_gold.count()}")

df_gold.show(10, truncate=False)


StatementMeta(, 3b6606d3-1092-494b-8b7e-363d2650cd6a, 5, Finished, Available, Finished, False)

Gold table created: dim_product
Records: 997
+-----------+----------+-----------------------------------+-----------+-------+--------------+
|product_key|product_id|product_name                       |category   |price  |stock_quantity|
+-----------+----------+-----------------------------------+-----------+-------+--------------+
|1          |1         |Centralized secondary application  |Fashion    |387.51 |479           |
|2          |2         |Implemented upward-trending concept|Fashion    |683.31 |154           |
|3          |3         |Secured full-range implementation  |Books      |180.83 |303           |
|4          |4         |Adaptive content-based algorithm   |Toys       |1695.86|10            |
|5          |5         |Versatile clear-thinking policy    |Toys       |863.54 |377           |
|6          |6         |Exclusive systemic ability         |Books      |1414.68|292           |
|7          |7         |Reduced global middleware          |Electronics|1272.47|92         